# Frozen Rater → Final Classifier — full diagnostics pipeline

This notebook preserves the existing frozen-Rater final-classifier experiment and adds the complete downstream diagnostics pipeline.

The final classifier is still a **fresh `Linear(2048, 2)`** trained on the same Waterbirds-ERM embeddings, using `val_subset1` for classifier training and `val_subset2` for checkpoint selection. The frozen Rater is used only to assign training weights; it is not part of inference.

Added outputs:

- train / validation / test **accuracy, WGA, mean CE loss, worst-group mean CE loss** over epochs;
- per-group accuracy and loss over epochs;
- selected-model metrics and per-group metrics;
- actual Rater-weight diagnostics during downstream training;
- `best.pt`, `latest.pt`, optional periodic checkpoints, and compact `best_final_classifier.pt`;
- selected-model Grad-CAM on fixed Waterbirds samples;
- clean CSV/JSON/log/summary artifacts compatible with the companion multi-seed plotting script.

In [ ]:
# ============================================================
# 1. Mount Drive + imports + list available frozen Raters
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
import copy, json, csv, random, math, os, re, shutil, subprocess, sys, time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DRIVE_ROOT = Path('/content/drive/MyDrive/evidential_alignment_colab')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

search_roots = [
    DRIVE_ROOT / 'rater_meta_only_runs',
    DRIVE_ROOT / 'rater_ea_gate_meta_runs',
    DRIVE_ROOT / 'rater_erm_minimal_runs',
    DRIVE_ROOT / 'rater_classwise_top_loss_runs'
]

rater_candidates = []
for root in search_roots:
    if not root.exists():
        continue
    for path in root.rglob('final_rater.pt'):
        try:
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
            if isinstance(ckpt, dict) and 'rater_sd' in ckpt:
                fp = ckpt.get('erm_fingerprint') or ckpt.get('erm_checkpoint_fingerprint')
                rater_candidates.append({
                    'path': path,
                    'mtime': path.stat().st_mtime,
                    'fingerprint': fp,
                    'capacity': ckpt.get('rater_capacity', '?'),
                    'weighting': ckpt.get('weighting') or ckpt.get('rater_weighting'),
                    'temperature': ckpt.get('temperature') or ckpt.get('rater_temperature'),
                })
        except Exception:
            pass

rater_candidates = sorted(rater_candidates, key=lambda x: x['mtime'], reverse=True)
print(f'Found {len(rater_candidates)} frozen Rater checkpoint(s):\n')
for i, item in enumerate(rater_candidates):
    print(f'[{i}] {item["path"]}')
    print('    fingerprint =', item['fingerprint'])
    print('    capacity =', item['capacity'], '| weighting =', item['weighting'], '| temperature =', item['temperature'])
    print()

In [ ]:
# ============================================================
# 2. EXPERIMENT CONFIGURATION
# ============================================================

# You can paste a full checkpoint path, or use:
# RATER_CHECKPOINT = rater_candidates[INDEX]['path']
# RATER_CHECKPOINT = Path(
#     '/content/drive/MyDrive/evidential_alignment_colab/'
#     'rater_classwise_top_loss_runs/'
#     'top30pct_per_class_all_sample_20261007_215941/'
#     'final_rater.pt'
# )

RATER_CHECKPOINT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab/'
    'rater_classwise_top_loss_runs/'
    'top10pct_per_class_all_sample_20261011_131220/'
    'final_rater.pt'
)
EMBEDDING_CACHE_ROOT = Path(
    '/content/drive/MyDrive/evidential_alignment_colab/'
    'embeddings_waterbirds_erm/waterbirds/'
    'resnet50_waterbirds_erm_1055b1e8de29/resolution_224'
)
EXPECTED_ERM_FINGERPRINT = '1055b1e8de29'

# ------------------------------------------------------------
# MAIN SWITCH
# ------------------------------------------------------------
# "all"           -> every sample gets a Rater-derived weight
# "misclassified" -> correct=CORRECT_WEIGHT; wrong=Rater weight
RATING_MODE = 'all'

# Same final-classifier split setup as your previous notebook.
SPLIT_VAL = 0.6
SEED = 2

# Final classifier hyperparameters.
BATCH_SIZE = 128
NUM_WORKERS = 2
FINAL_EPOCHS = 50
FINAL_LR = 1e-2
FINAL_MOMENTUM = 0.9
FINAL_WEIGHT_DECAY = 1e-4

# Rater -> weight hyperparameters.
# NORMALIZATION_MODE: 'global', 'batch', or 'none'
NORMALIZATION_MODE = 'global'
# WEIGHT_TRANSFORM: 'sigmoid' or 'softmax'
WEIGHT_TRANSFORM = 'sigmoid'
TEMPERATURE = 1.0
WEIGHT_SCALE = 1.0
CORRECT_WEIGHT = 1.0  # used only for RATING_MODE='misclassified'

# Optional ordinary CE added beside weighted CE. 0 = weighted CE only.
AUX_CE_COEFF = 0.0

# Model selection and plotting.
SELECTION_METRIC = 'wga'  # 'wga' or 'accuracy'
PLOT_EVERY = 1
EVAL_EVERY = 1

# Checkpoint policy. 0 = save only best + latest.
CHECKPOINT_EVERY = 0

# Selected-model Grad-CAM.
ENABLE_GRADCAM = True
GRADCAM_SPLIT = 'test'
GRADCAM_SAMPLES_PER_GROUP = 1

# Grad-CAM uses the original image-space ERM backbone.
REPO_DIR = Path('/content/evidential_alignment')
DATA_ROOT = Path('/content/datasets')
WB_DIR = DATA_ROOT / 'waterbird_complete95_forest2water2'
WB_ARCHIVE = DRIVE_ROOT / 'archives' / 'waterbird_complete95_forest2water2.tar.gz'
DEFAULT_ERM_MODEL = DRIVE_ROOT / 'erm_base_models' / 'waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt'
if not DEFAULT_ERM_MODEL.exists():
    DEFAULT_ERM_MODEL = DRIVE_ROOT / 'erm_base_models' / 'erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained' / 'best_val_acc_model.pt'

assert RATING_MODE in {'all', 'misclassified'}
assert NORMALIZATION_MODE in {'global', 'batch', 'none'}
assert WEIGHT_TRANSFORM in {'sigmoid', 'softmax'}
assert SELECTION_METRIC in {'wga', 'accuracy'}
assert TEMPERATURE > 0
assert EVAL_EVERY >= 1
assert PLOT_EVERY >= 1
assert CHECKPOINT_EVERY >= 0
assert RATER_CHECKPOINT.exists(), RATER_CHECKPOINT
assert EMBEDDING_CACHE_ROOT.exists(), EMBEDDING_CACHE_ROOT

RUNS_ROOT = DRIVE_ROOT / 'frozen_rater_final_runs_configurable'
RUN_NAME = (
    f'rating_{RATING_MODE}_norm_{NORMALIZATION_MODE}_{WEIGHT_TRANSFORM}'
    f'_temp_{TEMPERATURE}_lr_{FINAL_LR}_splitval_{SPLIT_VAL}_'
    f'{datetime.now().strftime("%Y%m%d_%H%M%S")}'
)
RUN_ROOT = RUNS_ROOT / RUN_NAME
PLOTS_ROOT = RUN_ROOT / 'plots'
DATA_ARTIFACT_ROOT = PLOTS_ROOT / 'data'
CHECKPOINT_ROOT = RUN_ROOT / 'checkpoints'
GRADCAM_ROOT = PLOTS_ROOT / 'gradcam'
RUN_ROOT.mkdir(parents=True, exist_ok=False)
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_ROOT / 'log.txt'
HISTORY_PATH = DATA_ARTIFACT_ROOT / 'history.csv'
METRICS_PATH = DATA_ARTIFACT_ROOT / 'metrics.csv'
GROUP_METRICS_PATH = DATA_ARTIFACT_ROOT / 'group_metrics.csv'
WEIGHT_STATS_PATH = DATA_ARTIFACT_ROOT / 'weight_stats.csv'

print('Frozen Rater:', RATER_CHECKPOINT)
print('Embedding cache:', EMBEDDING_CACHE_ROOT)
print('RATING_MODE:', RATING_MODE)
print('Run directory:', RUN_ROOT)

In [ ]:
# ============================================================
# 3. Reproducibility + clean logging / atomic artifact helpers
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
RUN_START_TIME = time.monotonic()


def log(msg=''):
    msg = str(msg)
    print(msg, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(msg + '\n')


def _json_value(value):
    if torch.is_tensor(value):
        return _json_value(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return _json_value(value.tolist())
    if isinstance(value, np.generic):
        return _json_value(value.item())
    if isinstance(value, dict):
        return {str(k): _json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_value(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + f'.tmp.{os.getpid()}')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(_json_value(payload), f, indent=2, sort_keys=True, allow_nan=False)
        f.write('\n')
    os.replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + f'.tmp.{os.getpid()}')
    torch.save(payload, tmp)
    os.replace(tmp, path)


def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def rng_state():
    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


config = {
    'stage': 'final_classifier',
    'rater_checkpoint': str(RATER_CHECKPOINT),
    'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    'expected_erm_fingerprint': EXPECTED_ERM_FINGERPRINT,
    'rating_mode': RATING_MODE,
    'split_val': SPLIT_VAL,
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'final_epochs': FINAL_EPOCHS,
    'final_lr': FINAL_LR,
    'final_momentum': FINAL_MOMENTUM,
    'final_weight_decay': FINAL_WEIGHT_DECAY,
    'normalization_mode': NORMALIZATION_MODE,
    'weight_transform': WEIGHT_TRANSFORM,
    'temperature': TEMPERATURE,
    'weight_scale': WEIGHT_SCALE,
    'correct_weight': CORRECT_WEIGHT,
    'aux_ce_coeff': AUX_CE_COEFF,
    'selection_metric': SELECTION_METRIC,
    'eval_every': EVAL_EVERY,
    'plot_every': PLOT_EVERY,
    'checkpoint_every': CHECKPOINT_EVERY,
    'gradcam': {
        'enabled': ENABLE_GRADCAM,
        'split': GRADCAM_SPLIT,
        'samples_per_group': GRADCAM_SAMPLES_PER_GROUP,
    },
}
write_json(RUN_ROOT / 'config.json', config)

log('=' * 100)
log('FROZEN RATER FINAL CLASSIFIER — FULL DIAGNOSTICS')
log('=' * 100)
log(json.dumps(config, indent=2))
log('=' * 100)

In [ ]:
# ============================================================
# 4. Load Waterbirds-ERM embedding splits
# ============================================================

class IndexedEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels, groups, attrs, split_name):
        self.embeddings = embeddings.float()
        self.labels = labels.long()
        self.groups = groups.long()
        self.attrs = attrs.long()
        self.split_name = split_name

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, idx):
        return idx, self.embeddings[idx], self.labels[idx], self.groups[idx], self.attrs[idx]


def _tok(x):
    return str(x).replace('.', 'p')


def split_path(split_name):
    if split_name in {'train_no_aug', 'val', 'test'}:
        return EMBEDDING_CACHE_ROOT / f'{split_name}.pt'
    if split_name in {'val_subset1', 'val_subset2'}:
        return EMBEDDING_CACHE_ROOT / (
            f'{split_name}_splittrain_1p0_splitval_{_tok(SPLIT_VAL)}_seed_{SEED}.pt'
        )
    raise ValueError(split_name)


def ensure_val_subsets():
    """Create requested split_val subsets directly from saved val.pt if absent."""
    p1 = split_path('val_subset1')
    p2 = split_path('val_subset2')
    if p1.exists() and p2.exists():
        log('[Embeddings] requested val subsets already exist.')
        return

    full_path = split_path('val')
    payload = torch.load(full_path, map_location='cpu', weights_only=False)
    n = int(payload['labels'].shape[0])

    rng = np.random.RandomState(SEED)
    indices = np.arange(n)
    rng.shuffle(indices)
    n1 = int(n * SPLIT_VAL)

    subset_indices = {
        'val_subset1': torch.as_tensor(indices[:n1], dtype=torch.long),
        'val_subset2': torch.as_tensor(indices[n1:], dtype=torch.long),
    }

    for split_name, idx in subset_indices.items():
        out_path = split_path(split_name)
        subset = {
            'embeddings': payload['embeddings'][idx].clone(),
            'labels': payload['labels'][idx].clone(),
            'groups': payload['groups'][idx].clone(),
            'attrs': payload['attrs'][idx].clone(),
            'cache_metadata': dict(payload.get('cache_metadata', {})),
        }
        subset['cache_metadata'].update({
            'split': split_name,
            'split_train': 1.0,
            'split_val': float(SPLIT_VAL),
            'seed': int(SEED),
            'derived_from_full_val_cache': True,
        })
        torch.save(subset, out_path)
        log(f'[Embeddings] created {split_name}: n={len(idx)}')


ensure_val_subsets()


def load_split(split_name):
    path = split_path(split_name)
    payload = torch.load(path, map_location='cpu', weights_only=False)
    z = payload['embeddings']
    assert z.ndim == 2 and z.shape[1] == 2048, (split_name, z.shape)

    metadata = payload.get('cache_metadata', {})
    representation = metadata.get('representation_source')
    if representation and representation != 'waterbirds_erm_checkpoint':
        raise RuntimeError(f'{split_name} is not a Waterbirds-ERM embedding cache.')

    fp = metadata.get('erm_checkpoint_fingerprint') or metadata.get('erm_fingerprint')
    if fp and fp != EXPECTED_ERM_FINGERPRINT:
        raise RuntimeError(f'{split_name} fingerprint mismatch: {fp}')

    ds = IndexedEmbeddingDataset(
        payload['embeddings'], payload['labels'], payload['groups'], payload['attrs'], split_name
    )
    log(f'[Embeddings] {split_name}: {len(ds)} samples')
    return ds


# Same roles as the previous final-classifier notebook.
calibration_split = 'val_subset1'
selection_split = 'val_subset2'
test_split = 'test'

calibration_dataset = load_split(calibration_split)
selection_dataset = load_split(selection_split)
test_dataset = load_split(test_split)

for name, ds in [('calibration', calibration_dataset), ('selection', selection_dataset), ('test', test_dataset)]:
    log('')
    log(f'{name}: n={len(ds)}')
    g = ds.groups.numpy()
    for gid in sorted(np.unique(g)):
        log(f'  g{gid}: {int((g == gid).sum())}')

# Canonical final-stage role names used by the logging pipeline.
training_split = calibration_split
validation_split = selection_split
datasets_by_role = {
    'train': calibration_dataset,
    'val': selection_dataset,
    'test': test_dataset,
}
split_names = {
    'train': calibration_split,
    'val': selection_split,
    'test': test_split,
}

In [ ]:
# ============================================================
# 5. Load and COMPLETELY FREEZE the selected Rater
# ============================================================

class EmbeddingRaterSmall(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )
    def forward(self, z):
        return self.net(z).squeeze(-1)


class EmbeddingRaterMedium(nn.Module):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )
    def forward(self, z):
        return self.net(z).squeeze(-1)


checkpoint = torch.load(RATER_CHECKPOINT, map_location='cpu', weights_only=False)
state_dict = checkpoint['rater_sd']
capacity = str(checkpoint.get('rater_capacity', '')).lower()

if capacity not in {'small', 'medium'}:
    first = state_dict.get('net.1.weight')
    if first is not None and first.shape[0] == 256:
        capacity = 'small'
    elif first is not None and first.shape[0] == 512:
        capacity = 'medium'
    else:
        raise RuntimeError('Could not infer Rater architecture.')

rater = EmbeddingRaterSmall(2048) if capacity == 'small' else EmbeddingRaterMedium(2048)
rater.load_state_dict(state_dict)
rater.to(DEVICE)
rater.eval()
for p in rater.parameters():
    p.requires_grad_(False)
assert not any(p.requires_grad for p in rater.parameters())

checkpoint_fp = checkpoint.get('erm_fingerprint') or checkpoint.get('erm_checkpoint_fingerprint')
if checkpoint_fp and checkpoint_fp != EXPECTED_ERM_FINGERPRINT:
    raise RuntimeError(
        f'Rater fingerprint {checkpoint_fp} != embedding fingerprint {EXPECTED_ERM_FINGERPRINT}'
    )

log('')
log('Frozen Rater loaded:')
log(f'  path = {RATER_CHECKPOINT}')
log(f'  capacity = {capacity}')
log(f'  ERM fingerprint = {checkpoint_fp}')
log(f'  Rater training weighting = {checkpoint.get("weighting") or checkpoint.get("rater_weighting")}')
log(f'  Rater training temperature = {checkpoint.get("temperature") or checkpoint.get("rater_temperature")}')
log('  trainable Rater parameters = 0')

RATER_SIGNATURE = {
    'capacity': capacity,
    'erm_fingerprint': checkpoint_fp,
    'training_weighting': checkpoint.get('weighting') or checkpoint.get('rater_weighting'),
    'training_temperature': checkpoint.get('temperature') or checkpoint.get('rater_temperature'),
    'outer_loss_type': checkpoint.get('outer_loss_type') or checkpoint.get('meta_objective'),
    'outer_top_loss_percent_per_class': checkpoint.get('outer_top_loss_percent_per_class'),
    'outer_lse_beta': checkpoint.get('outer_lse_beta'),
    'best_meta_step': checkpoint.get('best_meta_step'),
    'best_outer_loss': checkpoint.get('best_outer_loss'),
}
config['rater_signature'] = RATER_SIGNATURE
write_json(RUN_ROOT / 'config.json', config)

In [6]:
# ============================================================
# 6. Precompute frozen-Rater raw scores once
# ============================================================

@torch.no_grad()
def precompute_raw_scores(dataset):
    loader = DataLoader(
        dataset, batch_size=512, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    scores = torch.empty(len(dataset), dtype=torch.float32)
    rater.eval()

    for idx, z, _, _, _ in loader:
        z = z.to(DEVICE, non_blocking=True)
        scores[idx.long()] = rater(z).detach().cpu()

    return scores


calibration_raw_scores = precompute_raw_scores(calibration_dataset)
selection_raw_scores = precompute_raw_scores(selection_dataset)
test_raw_scores = precompute_raw_scores(test_dataset)

GLOBAL_RATER_MEAN = float(calibration_raw_scores.mean())
GLOBAL_RATER_STD = float(calibration_raw_scores.std(unbiased=False))

log('')
log(f'Global calibration Rater mean = {GLOBAL_RATER_MEAN:.8f}')
log(f'Global calibration Rater std  = {GLOBAL_RATER_STD:.8f}')

# Raw Rater distribution by Waterbirds group.
fig, ax = plt.subplots(figsize=(10, 7))
groups = calibration_dataset.groups.numpy()
raw = calibration_raw_scores.numpy()
for gid in sorted(np.unique(groups)):
    mask = groups == gid
    ax.hist(raw[mask], bins=40, density=True, alpha=0.45, label=f'g{gid} (n={int(mask.sum())})')
ax.set_xlabel('Frozen Rater raw score')
ax.set_ylabel('Density')
ax.set_title('Frozen Rater raw scores on calibration set')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'calibration_raw_rater_scores_by_group.png', dpi=160, bbox_inches='tight')
plt.close(fig)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Global calibration Rater mean = -0.06717836
Global calibration Rater std  = 0.05202202


In [7]:
# ============================================================
# 7. Configurable Rater-to-weight rule
# ============================================================

def normalize_scores(raw_scores):
    if NORMALIZATION_MODE == 'global':
        return (raw_scores - GLOBAL_RATER_MEAN) / (GLOBAL_RATER_STD + 1e-6)

    if NORMALIZATION_MODE == 'batch':
        mean = raw_scores.mean()
        std = raw_scores.std(unbiased=False)
        return (raw_scores - mean) / (std + 1e-6)

    if NORMALIZATION_MODE == 'none':
        return raw_scores

    raise ValueError(NORMALIZATION_MODE)


def transform_scores(normalized_scores):
    if WEIGHT_TRANSFORM == 'sigmoid':
        return WEIGHT_SCALE * torch.sigmoid(normalized_scores / TEMPERATURE)

    if WEIGHT_TRANSFORM == 'softmax':
        centered = normalized_scores - normalized_scores.mean()
        return WEIGHT_SCALE * torch.softmax(centered / TEMPERATURE, dim=0)

    raise ValueError(WEIGHT_TRANSFORM)


@torch.no_grad()
def compute_training_weights(batch_indices, logits, y):
    """
    RATING_MODE='all':
        every sample gets a transformed Rater weight.

    RATING_MODE='misclassified':
        correct samples get CORRECT_WEIGHT;
        wrong samples get transformed Rater weights.
    """
    idx_cpu = batch_indices.detach().cpu().long()
    raw_all = calibration_raw_scores[idx_cpu].to(logits.device)

    pred = logits.argmax(dim=1)
    correct_mask = pred.eq(y)
    wrong_mask = ~correct_mask

    if RATING_MODE == 'all':
        weights = transform_scores(normalize_scores(raw_all))
        return weights, correct_mask, wrong_mask, raw_all

    # Misclassified-only mode.
    weights = torch.full(
        (y.shape[0],), float(CORRECT_WEIGHT),
        device=logits.device, dtype=logits.dtype,
    )

    if bool(wrong_mask.any()):
        raw_wrong = raw_all[wrong_mask]

        if NORMALIZATION_MODE == 'global':
            normalized_wrong = (raw_wrong - GLOBAL_RATER_MEAN) / (GLOBAL_RATER_STD + 1e-6)
        elif NORMALIZATION_MODE == 'batch':
            normalized_wrong = (
                raw_wrong - raw_wrong.mean()
            ) / (
                raw_wrong.std(unbiased=False) + 1e-6
            )
        elif NORMALIZATION_MODE == 'none':
            normalized_wrong = raw_wrong
        else:
            raise ValueError(NORMALIZATION_MODE)

        wrong_weights = transform_scores(normalized_wrong)
        wrong_idx = torch.nonzero(wrong_mask, as_tuple=False).squeeze(1)
        weights = weights.index_copy(0, wrong_idx, wrong_weights)

    return weights, correct_mask, wrong_mask, raw_all

In [ ]:
# ============================================================
# 8. Fresh final classifier + complete downstream diagnostics
# ============================================================

METRIC_FIELDS = ('accuracy', 'wga', 'loss', 'worst_group_loss')
METRIC_LABELS = ('Accuracy', 'Worst-group accuracy', 'Mean CE loss', 'Worst-group mean CE loss')


class FinalLinearClassifier(nn.Module):
    def __init__(self, input_dim=2048, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, z):
        return self.linear(z)


@torch.no_grad()
def evaluate_classifier(model, dataset):
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_n = 0
    group_correct = {}
    group_total = {}
    group_loss_sum = {}

    for _, z, y, g, _ in loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)

        logits = model(z)
        per_loss = F.cross_entropy(logits, y, reduction='none')
        pred = logits.argmax(dim=1)

        total_loss += float(per_loss.sum().item())
        total_correct += int(pred.eq(y).sum().item())
        total_n += int(y.numel())

        for gid_t in torch.unique(g):
            gid = int(gid_t.item())
            mask = g.eq(gid_t)
            group_correct[gid] = group_correct.get(gid, 0) + int(pred[mask].eq(y[mask]).sum().item())
            group_total[gid] = group_total.get(gid, 0) + int(mask.sum().item())
            group_loss_sum[gid] = group_loss_sum.get(gid, 0.0) + float(per_loss[mask].sum().item())

    if total_n == 0:
        raise RuntimeError('Cannot evaluate an empty dataset.')

    group_accuracy = {gid: group_correct[gid] / group_total[gid] for gid in sorted(group_total)}
    group_loss = {gid: group_loss_sum[gid] / group_total[gid] for gid in sorted(group_total)}

    return {
        'n_samples': total_n,
        'loss': total_loss / total_n,
        'accuracy': total_correct / total_n,
        'group_accuracy': group_accuracy,
        'group_loss': group_loss,
        'group_count': group_total,
        'wga': min(group_accuracy.values()),
        'worst_group_loss': max(group_loss.values()),
    }


def selection_value(metrics):
    if SELECTION_METRIC == 'wga':
        return metrics['wga']
    if SELECTION_METRIC == 'accuracy':
        return metrics['accuracy']
    raise ValueError(SELECTION_METRIC)


def append_csv_rows(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists() and path.stat().st_size > 0
    with open(path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        if not exists:
            writer.writeheader()
        writer.writerows(rows)


metric_rows = []
group_metric_rows = []
weight_rows = []


def record_metrics(epoch, role, metrics, phase='trajectory'):
    row = {
        'seed': SEED,
        'stage': 'final_classifier',
        'phase': phase,
        'epoch': int(epoch),
        'elapsed_seconds': time.monotonic() - RUN_START_TIME,
        'split': split_names[role],
        'role': role,
        'n_samples': metrics['n_samples'],
        'accuracy': metrics['accuracy'],
        'wga': metrics['wga'],
        'loss': metrics['loss'],
        'worst_group_loss': metrics['worst_group_loss'],
    }
    metric_rows.append(row)
    append_csv_rows(METRICS_PATH, [row])

    rows = []
    for gid in sorted(metrics['group_count']):
        rows.append({
            'seed': SEED,
            'stage': 'final_classifier',
            'phase': phase,
            'epoch': int(epoch),
            'split': split_names[role],
            'role': role,
            'group': int(gid),
            'n_samples': int(metrics['group_count'][gid]),
            'accuracy': metrics['group_accuracy'][gid],
            'loss': metrics['group_loss'][gid],
        })
    group_metric_rows.extend(rows)
    append_csv_rows(GROUP_METRICS_PATH, rows)


def log_metrics(epoch, by_role, prefix='epoch'):
    chunks = []
    for role in ('train', 'val', 'test'):
        m = by_role[role]
        chunks.append(
            f'{role}: acc={100*m["accuracy"]:.2f}% WGA={100*m["wga"]:.2f}% '
            f'loss={m["loss"]:.6f} worst_group_loss={m["worst_group_loss"]:.6f}'
        )
    log(f'[{prefix} {epoch:03d}] ' + ' | '.join(chunks))


def save_metric_curves():
    rows = [r for r in metric_rows if r['phase'] == 'trajectory']
    if not rows:
        return
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, field, label in zip(axes.flat, METRIC_FIELDS, METRIC_LABELS):
        for role in ('train', 'val', 'test'):
            vals = sorted([r for r in rows if r['role'] == role], key=lambda r: r['epoch'])
            ax.plot([r['epoch'] for r in vals], [r[field] for r in vals], marker='.', label=role)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=9)
    fig.suptitle('Final classifier: train / validation / test metrics over time')
    fig.tight_layout()
    fig.savefig(PLOTS_ROOT / 'final_metrics_over_time.png', dpi=160, bbox_inches='tight')
    plt.close(fig)


def save_group_metric_curves():
    rows = [r for r in group_metric_rows if r['phase'] == 'trajectory']
    for role in ('train', 'val', 'test'):
        role_rows = [r for r in rows if r['role'] == role]
        if not role_rows:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, field, ylabel in ((axes[0], 'accuracy', 'Group accuracy'), (axes[1], 'loss', 'Group mean CE')):
            for gid in sorted({r['group'] for r in role_rows}):
                vals = sorted([r for r in role_rows if r['group'] == gid], key=lambda r: r['epoch'])
                ax.plot([r['epoch'] for r in vals], [r[field] for r in vals], marker='.', label=f'Group {gid}')
            ax.set_xlabel('Epoch')
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=9)
        fig.suptitle(f'Final classifier per-group metrics: {role}')
        fig.tight_layout()
        fig.savefig(PLOTS_ROOT / f'group_metrics_{role}_over_time.png', dpi=160, bbox_inches='tight')
        plt.close(fig)


def _bins(values):
    values = np.asarray(values, dtype=np.float64)
    finite = values[np.isfinite(values)]
    lo, hi = float(finite.min()), float(finite.max())
    if np.isclose(lo, hi):
        eps = max(abs(lo) * 0.05, 1e-6)
        return np.linspace(lo - eps, hi + eps, 20)
    return np.linspace(lo, hi, 41)


def save_weight_plots(epoch, weights, groups, correctness):
    weights = np.asarray(weights, dtype=np.float64)
    groups = np.asarray(groups, dtype=np.int64)
    correctness = np.asarray(correctness) > 0.5
    plot_dir = PLOTS_ROOT / 'weight_histograms'
    plot_dir.mkdir(parents=True, exist_ok=True)
    bins = _bins(weights)

    # Requested 5-panel group layout: four separate left panels + large overlay right.
    fig = plt.figure(figsize=(16, 12))
    grid = fig.add_gridspec(4, 2, width_ratios=[1.0, 2.15], hspace=0.50, wspace=0.28)
    for row, gid in enumerate((0, 1, 2, 3)):
        ax = fig.add_subplot(grid[row, 0])
        mask = groups == gid
        if np.any(mask):
            ax.hist(weights[mask], bins=bins, density=True, alpha=0.80)
        ax.set_xlim(bins[0], bins[-1])
        ax.set_ylabel('Density')
        ax.set_title(f'Group {gid} (n={int(mask.sum())})')
        ax.grid(True, linestyle=':', alpha=0.25)
        if row == 3:
            ax.set_xlabel('Actual final-classifier training weight')

    ax_all = fig.add_subplot(grid[:, 1])
    for gid in (0, 1, 2, 3):
        mask = groups == gid
        if np.any(mask):
            ax_all.hist(weights[mask], bins=bins, density=True, alpha=0.38,
                        label=f'Group {gid} (n={int(mask.sum())})')
    ax_all.set_xlim(bins[0], bins[-1])
    ax_all.set_xlabel('Actual final-classifier training weight')
    ax_all.set_ylabel('Density')
    ax_all.set_title('All groups overlaid')
    ax_all.grid(True, linestyle=':', alpha=0.25)
    ax_all.legend(fontsize=9)
    fig.suptitle(f'Final-classifier weights | epoch {epoch} | mode={RATING_MODE}', fontsize=15)
    fig.subplots_adjust(top=0.94)
    fig.savefig(plot_dir / f'epoch_{epoch:03d}_weights_by_group.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 7))
    if correctness.any():
        ax.hist(weights[correctness], bins=bins, density=True, alpha=0.50,
                label=f'correct (n={int(correctness.sum())})')
    wrong = ~correctness
    if wrong.any():
        ax.hist(weights[wrong], bins=bins, density=True, alpha=0.50,
                label=f'wrong (n={int(wrong.sum())})')
    ax.set_xlabel('Actual training weight')
    ax.set_ylabel('Density')
    ax.set_title(f'Epoch {epoch}: weights by current correctness')
    ax.grid(True, linestyle=':', alpha=0.30)
    ax.legend()
    fig.tight_layout()
    fig.savefig(plot_dir / f'epoch_{epoch:03d}_weights_correct_vs_wrong.png', dpi=160, bbox_inches='tight')
    plt.close(fig)


def record_weight_stats(epoch, weights, groups, correctness):
    weights = np.asarray(weights, dtype=np.float64)
    groups = np.asarray(groups, dtype=np.int64)
    correctness = np.asarray(correctness) > 0.5
    masks = {
        'all': np.ones(len(weights), dtype=bool),
        'correct': correctness,
        'wrong': ~correctness,
    }
    for gid in sorted(np.unique(groups)):
        masks[f'group_{gid}'] = groups == gid
    rows = []
    for subset, mask in masks.items():
        vals = weights[mask]
        if len(vals) == 0:
            continue
        rows.append({
            'seed': SEED, 'epoch': int(epoch), 'subset': subset, 'n': int(len(vals)),
            'mean': float(vals.mean()), 'std': float(vals.std()), 'min': float(vals.min()),
            'q25': float(np.quantile(vals, .25)), 'median': float(np.median(vals)),
            'q75': float(np.quantile(vals, .75)), 'max': float(vals.max()),
            'mass_share': float(vals.sum() / max(float(weights.sum()), 1e-12)),
        })
    weight_rows.extend(rows)
    append_csv_rows(WEIGHT_STATS_PATH, rows)


def save_selected_model_plots(selected_metrics):
    roles = ('train', 'val', 'test')
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, field, label in zip(axes.flat, METRIC_FIELDS, METRIC_LABELS):
        ax.bar(roles, [selected_metrics[r][field] for r in roles])
        ax.set_ylabel(label)
        ax.grid(True, axis='y', alpha=0.25)
    fig.suptitle('Selected final classifier')
    fig.tight_layout()
    fig.savefig(PLOTS_ROOT / 'selected_model_overall_metrics.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

    for role in roles:
        m = selected_metrics[role]
        gids = sorted(m['group_accuracy'])
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].bar([f'g{g}' for g in gids], [m['group_accuracy'][g] for g in gids])
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title(f'{role}: group accuracy')
        axes[0].grid(True, axis='y', alpha=0.25)
        axes[1].bar([f'g{g}' for g in gids], [m['group_loss'][g] for g in gids])
        axes[1].set_ylabel('Mean CE')
        axes[1].set_title(f'{role}: group loss')
        axes[1].grid(True, axis='y', alpha=0.25)
        fig.tight_layout()
        fig.savefig(PLOTS_ROOT / f'selected_model_groups_{role}.png', dpi=160, bbox_inches='tight')
        plt.close(fig)

In [ ]:
# ============================================================
# 9. Train final classifier + record requested metrics over time
# ============================================================

set_seed(SEED)
final_classifier = FinalLinearClassifier(2048, 2).to(DEVICE)
optimizer = torch.optim.SGD(
    final_classifier.parameters(),
    lr=FINAL_LR,
    momentum=FINAL_MOMENTUM,
    weight_decay=FINAL_WEIGHT_DECAY,
)

train_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


def checkpoint_payload(epoch, metrics_by_role, kind):
    return {
        'schema_version': 2,
        'stage': 'final_classifier',
        'checkpoint_kind': kind,
        'epoch': int(epoch),
        'model_sd': cpu_state_dict(final_classifier),
        'optimizer_sd': copy.deepcopy(optimizer.state_dict()),
        'rng_state': rng_state(),
        'config': config,
        'metrics': metrics_by_role,
        'selection_metric': SELECTION_METRIC,
        'rater_checkpoint': str(RATER_CHECKPOINT),
        'rater_signature': RATER_SIGNATURE,
        'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
        'erm_fingerprint': EXPECTED_ERM_FINGERPRINT,
    }


# Epoch 0 gives the true fresh-classifier starting point for all metric curves.
initial_metrics = {role: evaluate_classifier(final_classifier, ds) for role, ds in datasets_by_role.items()}
for role, m in initial_metrics.items():
    record_metrics(0, role, m, phase='trajectory')
log_metrics(0, initial_metrics, prefix='initial')

best_value = -float('inf')
best_epoch = None
history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    final_classifier.train()

    weighted_objective_sum = 0.0
    unweighted_ce_sum = 0.0
    seen = 0
    epoch_weights = []
    epoch_groups = []
    epoch_correctness = []

    for batch_indices, z, y, g, _ in train_loader:
        z = z.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g = g.to(DEVICE, non_blocking=True)

        logits = final_classifier(z)
        weights, correct_mask, _, _ = compute_training_weights(batch_indices, logits, y)
        per_sample_ce = F.cross_entropy(logits, y, reduction='none')
        weighted_ce = (per_sample_ce * weights).sum()
        ordinary_ce = per_sample_ce.mean()
        objective = weighted_ce + AUX_CE_COEFF * ordinary_ce

        optimizer.zero_grad()
        objective.backward()
        optimizer.step()

        weighted_objective_sum += float(weighted_ce.item())
        unweighted_ce_sum += float(per_sample_ce.sum().item())
        seen += int(y.numel())
        epoch_weights.append(weights.detach().cpu())
        epoch_groups.append(g.detach().cpu())
        epoch_correctness.append(correct_mask.float().detach().cpu())

    weights_np = torch.cat(epoch_weights).numpy()
    groups_np = torch.cat(epoch_groups).numpy()
    correctness_np = torch.cat(epoch_correctness).numpy()

    record_weight_stats(epoch, weights_np, groups_np, correctness_np)
    if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == FINAL_EPOCHS:
        save_weight_plots(epoch, weights_np, groups_np, correctness_np)

    # Clean, unweighted evaluation AFTER the epoch on train/val/test.
    metrics_by_role = {role: evaluate_classifier(final_classifier, ds) for role, ds in datasets_by_role.items()}
    for role, m in metrics_by_role.items():
        record_metrics(epoch, role, m, phase='trajectory')

    current_value = selection_value(metrics_by_role['val'])
    row = {
        'seed': SEED,
        'epoch': epoch,
        'optimization_weighted_loss_sum_per_epoch': weighted_objective_sum,
        'optimization_unweighted_mean_ce_seen_during_training': unweighted_ce_sum / max(seen, 1),
        'weight_mean': float(weights_np.mean()),
        'weight_std': float(weights_np.std()),
        'train_accuracy': metrics_by_role['train']['accuracy'],
        'train_wga': metrics_by_role['train']['wga'],
        'train_loss': metrics_by_role['train']['loss'],
        'train_worst_group_loss': metrics_by_role['train']['worst_group_loss'],
        'val_accuracy': metrics_by_role['val']['accuracy'],
        'val_wga': metrics_by_role['val']['wga'],
        'val_loss': metrics_by_role['val']['loss'],
        'val_worst_group_loss': metrics_by_role['val']['worst_group_loss'],
        'test_accuracy': metrics_by_role['test']['accuracy'],
        'test_wga': metrics_by_role['test']['wga'],
        'test_loss': metrics_by_role['test']['loss'],
        'test_worst_group_loss': metrics_by_role['test']['worst_group_loss'],
    }
    history.append(row)
    append_csv_rows(HISTORY_PATH, [row])

    # Save latest every epoch.
    latest = checkpoint_payload(epoch, metrics_by_role, 'latest')
    latest.update(selection_value=float(current_value), best_epoch_so_far=best_epoch,
                  best_value_so_far=(best_value if math.isfinite(best_value) else None))
    atomic_torch_save(latest, CHECKPOINT_ROOT / 'latest.pt')

    # Best checkpoint is selected ONLY by val_subset2.
    if current_value > best_value:
        best_value = float(current_value)
        best_epoch = int(epoch)
        best = checkpoint_payload(epoch, metrics_by_role, 'best_validation_selected')
        best['selection_value'] = best_value
        atomic_torch_save(best, CHECKPOINT_ROOT / 'best.pt')

    if CHECKPOINT_EVERY > 0 and epoch % CHECKPOINT_EVERY == 0:
        snap = checkpoint_payload(epoch, metrics_by_role, 'periodic')
        snap['selection_value'] = float(current_value)
        atomic_torch_save(snap, CHECKPOINT_ROOT / f'epoch_{epoch:03d}.pt')

    log(f'[epoch {epoch:03d}/{FINAL_EPOCHS}] weighted_objective_sum={weighted_objective_sum:.6f} '
        f'weight_mean={weights_np.mean():.6f}')
    log_metrics(epoch, metrics_by_role)

    if epoch == 1 or epoch % EVAL_EVERY == 0 or epoch == FINAL_EPOCHS:
        save_metric_curves()
        save_group_metric_curves()

if best_epoch is None:
    raise RuntimeError('No validation-selected classifier checkpoint was produced.')

log('')
log(f'Validation-selected epoch = {best_epoch}')
log(f'Best {SELECTION_METRIC} = {best_value:.8f}')

In [ ]:
# ============================================================
# 10. Restore selected model + selected metrics / plots / summary
# ============================================================

best_checkpoint = torch.load(CHECKPOINT_ROOT / 'best.pt', map_location='cpu', weights_only=False)
final_classifier.load_state_dict(best_checkpoint['model_sd'])
final_classifier.to(DEVICE)
final_classifier.eval()

selected_metrics = {role: evaluate_classifier(final_classifier, ds) for role, ds in datasets_by_role.items()}
for role, m in selected_metrics.items():
    record_metrics(best_epoch, role, m, phase='selected')

log('')
log('=' * 100)
log(f'SELECTED FINAL CLASSIFIER (epoch {best_epoch})')
log_metrics(best_epoch, selected_metrics, prefix='selected')
log('=' * 100)

save_metric_curves()
save_group_metric_curves()
save_selected_model_plots(selected_metrics)

compact = {
    'schema_version': 2,
    'stage': 'final_classifier',
    'model_sd': cpu_state_dict(final_classifier),
    'selected_epoch': best_epoch,
    'selection_metric': SELECTION_METRIC,
    'selection_value': best_value,
    'selected_metrics': selected_metrics,
    'rating_mode': RATING_MODE,
    'normalization_mode': NORMALIZATION_MODE,
    'weight_transform': WEIGHT_TRANSFORM,
    'temperature': TEMPERATURE,
    'weight_scale': WEIGHT_SCALE,
    'correct_weight': CORRECT_WEIGHT,
    'aux_ce_coeff': AUX_CE_COEFF,
    'rater_checkpoint': str(RATER_CHECKPOINT),
    'rater_signature': RATER_SIGNATURE,
    'embedding_cache_root': str(EMBEDDING_CACHE_ROOT),
    'erm_fingerprint': EXPECTED_ERM_FINGERPRINT,
}
atomic_torch_save(compact, RUN_ROOT / 'best_final_classifier.pt')
write_json(DATA_ARTIFACT_ROOT / 'selected_metrics.json', selected_metrics)

summary = {
    'schema_version': 2,
    'status': 'complete_before_gradcam',
    'stage': 'final_classifier',
    'seed': SEED,
    'selected_epoch': best_epoch,
    'selection_metric': SELECTION_METRIC,
    'selection_value': best_value,
    'selected_metrics': selected_metrics,
    'initial_metrics': initial_metrics,
    'rater_signature': RATER_SIGNATURE,
    'rating_mode': RATING_MODE,
    'elapsed_seconds': time.monotonic() - RUN_START_TIME,
    'splits': split_names,
    'split_sizes': {role: len(ds) for role, ds in datasets_by_role.items()},
    'metric_definition': {
        'accuracy': 'fraction correct',
        'wga': 'minimum accuracy over observed Waterbirds groups',
        'loss': 'unweighted sample-mean cross entropy',
        'worst_group_loss': 'maximum mean cross entropy over observed Waterbirds groups',
        'train_role': 'val_subset1 used to fit the final classifier',
        'val_role': 'val_subset2 used to select the checkpoint',
        'test_role': 'diagnostic/reporting only; never used for checkpoint selection',
    },
}
write_json(RUN_ROOT / 'summary.json', summary)

with open(RUN_ROOT / 'summary.txt', 'w', encoding='utf-8') as f:
    f.write('FINAL CLASSIFIER SUMMARY\n' + '=' * 92 + '\n')
    f.write(f'Seed: {SEED}\n')
    f.write(f'Rater: {RATER_CHECKPOINT}\n')
    f.write(f'ERM fingerprint: {EXPECTED_ERM_FINGERPRINT}\n')
    f.write(f'Rating mode: {RATING_MODE}\n')
    f.write(f'Selected epoch: {best_epoch}\n')
    f.write(f'Selection metric: {SELECTION_METRIC}\n')
    f.write(f'Selection value: {best_value:.8f}\n\n')
    f.write('SELECTED MODEL METRICS\n' + '-' * 92 + '\n')
    for role in ('train', 'val', 'test'):
        m = selected_metrics[role]
        f.write(f'{role}={split_names[role]}: accuracy={m["accuracy"]:.8f}, WGA={m["wga"]:.8f}, '
                f'loss={m["loss"]:.8f}, worst_group_loss={m["worst_group_loss"]:.8f}\n')
        for gid in sorted(m['group_accuracy']):
            f.write(f'  g{gid}: n={m["group_count"][gid]}, accuracy={m["group_accuracy"][gid]:.8f}, '
                    f'loss={m["group_loss"][gid]:.8f}\n')
    f.write('\nARTIFACTS\n' + '-' * 92 + '\n')
    f.write('best_final_classifier.pt\ncheckpoints/best.pt\ncheckpoints/latest.pt\n')
    f.write('plots/final_metrics_over_time.png\nplots/group_metrics_*_over_time.png\n')
    f.write('plots/selected_model_*.png\nplots/weight_histograms/\nplots/gradcam/\n')
    f.write('plots/data/metrics.csv\nplots/data/group_metrics.csv\nplots/data/weight_stats.csv\nplots/data/history.csv\n')
    f.write('log.txt\n')

print('Selected model restored and summarized.')
print('Run directory:', RUN_ROOT)

In [ ]:
# ============================================================
# 11. Grad-CAM for the SELECTED final classifier
# ============================================================
# Final inference is image -> frozen ERM backbone -> selected Linear(2048,2).
# The Rater is NOT part of inference, so Grad-CAM targets the selected class logit.


def _prepare_gradcam_repo_and_data():
    if not REPO_DIR.exists():
        subprocess.run([
            'git', 'clone',
            'https://github.com/TheRealMmd/evidential_alignment.git',
            str(REPO_DIR),
        ], check=True)

    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'timm', 'transformers', 'wilds', 'wget',
    ], check=True)

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    WB_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    if not (WB_ARCHIVE.exists() and WB_ARCHIVE.stat().st_size > 100_000_000):
        subprocess.run([
            'wget', '-c',
            'https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz',
            '-O', str(WB_ARCHIVE),
        ], check=True)
    if not (WB_DIR / 'metadata.csv').exists():
        subprocess.run(['tar', '-xzf', str(WB_ARCHIVE), '-C', str(DATA_ROOT)], check=True)

    config_path = REPO_DIR / 'config.py'
    text = config_path.read_text(encoding='utf-8')
    placeholder = '"waterbirds": "/path/to/datasets/waterbirds"'
    if placeholder in text:
        text = text.replace(placeholder, f'"waterbirds": "{WB_DIR}"')
    else:
        text, n = re.subn(
            r'("waterbirds"\\s*:\\s*)["\\\'][^"\\\']*["\\\']',
            lambda m: m.group(1) + '"' + str(WB_DIR) + '"',
            text,
            count=1,
        )
        if n != 1:
            raise RuntimeError('Could not patch Waterbirds path in config.py')
    config_path.write_text(text, encoding='utf-8')


def _denormalize_image(x, dataset):
    display = x.detach().cpu()[0].clone()
    transforms_list = list(getattr(getattr(dataset, 'transform', None), 'transforms', []))
    for transform in reversed(transforms_list):
        if hasattr(transform, 'mean') and hasattr(transform, 'std'):
            mean = torch.as_tensor(transform.mean, dtype=display.dtype)[:, None, None]
            std = torch.as_tensor(transform.std, dtype=display.dtype)[:, None, None]
            display = display * std + mean
            break
    return display.clamp(0, 1).permute(1, 2, 0).numpy()


def save_selected_gradcam():
    if not ENABLE_GRADCAM:
        log('[Grad-CAM] disabled.')
        return []

    _prepare_gradcam_repo_and_data()
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    import data  # triggers dataset/transform registration
    from data.data_utils import prepare_data
    from models.classifier import Classifier

    image_config = SimpleNamespace(
        dataset='waterbirds', resolution=224,
        split_train=1.0, split_val=float(SPLIT_VAL), seed=int(SEED),
        train_split='train', batch_size=int(BATCH_SIZE), num_workers=0,
    )
    image_datasets, _ = prepare_data(image_config)
    if GRADCAM_SPLIT not in image_datasets:
        raise ValueError(f'Grad-CAM split unavailable: {GRADCAM_SPLIT}')
    image_dataset = image_datasets[GRADCAM_SPLIT]

    recorded_erm = checkpoint.get('erm_model_path')
    erm_model_path = Path(recorded_erm) if recorded_erm and Path(recorded_erm).exists() else DEFAULT_ERM_MODEL
    if not erm_model_path.exists():
        raise FileNotFoundError(erm_model_path)

    feature_model = Classifier(backbone='resnet50', num_classes=2, pretrained=False)
    erm_ckpt = torch.load(erm_model_path, map_location='cpu', weights_only=False)
    erm_sd = erm_ckpt['model_sd'] if isinstance(erm_ckpt, dict) and 'model_sd' in erm_ckpt else erm_ckpt
    erm_sd = {k: v for k, v in erm_sd.items() if not k.endswith('position_ids')}
    feature_model.load_state_dict(erm_sd, strict=False)
    backbone = feature_model.backbone.cpu().eval()
    head = copy.deepcopy(final_classifier).cpu().eval()

    target_layer = getattr(backbone, 'layer4', None)
    if target_layer is None:
        raise RuntimeError('ResNet backbone has no layer4 for Grad-CAM.')

    GRADCAM_ROOT.mkdir(parents=True, exist_ok=True)
    activations = []
    hook = target_layer.register_forward_hook(lambda module, args, output: activations.append(output))
    groups = np.asarray(image_dataset.group_array)
    sample_indices = []
    for gid in sorted(np.unique(groups)):
        candidates = np.flatnonzero(groups == gid)
        sample_indices.extend([int(i) for i in candidates[:GRADCAM_SAMPLES_PER_GROUP]])

    manifest = []
    try:
        for index in sample_indices:
            x, y, group, attr = image_dataset[index]
            x = x.detach().cpu().unsqueeze(0).requires_grad_(True)
            activations.clear()
            with torch.enable_grad():
                embedding = backbone(x)
                activation = activations[-1]
                logits = head(embedding)
                probs = torch.softmax(logits, dim=1)
                pred = int(logits.argmax(dim=1).item())
                target = logits[0, pred]
                grads = torch.autograd.grad(target, activation)[0]
                weights = grads.mean(dim=(2, 3), keepdim=True)
                cam = F.relu((weights * activation).sum(dim=1))
                cam = F.interpolate(cam[:, None], size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
                cam = cam.detach()
                cam = (cam - cam.min()) / (cam.max() - cam.min()).clamp_min(1e-12)
                cam_np = cam.numpy()

            display = _denormalize_image(x, image_dataset)
            filename_array = getattr(image_dataset, 'filename_array', None)
            filename = str(filename_array[index]) if filename_array is not None else None
            stem = f'{GRADCAM_SPLIT}_g{int(group)}_sample{index:06d}'

            fig, axes = plt.subplots(1, 2, figsize=(9, 4))
            axes[0].imshow(display)
            axes[0].set_title(f'Original\\ny={int(y)}, group={int(group)}')
            axes[1].imshow(display)
            axes[1].imshow(cam_np, alpha=0.45)
            axes[1].set_title(f'Selected final classifier Grad-CAM\\npred={pred}, p={float(probs[0, pred]):.3f}')
            for ax in axes:
                ax.axis('off')
            fig.tight_layout()
            fig.savefig(GRADCAM_ROOT / f'{stem}.png', dpi=150, bbox_inches='tight')
            plt.close(fig)

            np.savez_compressed(
                GRADCAM_ROOT / f'{stem}.npz',
                image=display, cam=cam_np,
                logits=logits.detach().numpy(),
                probabilities=probs.detach().numpy(),
            )
            manifest.append({
                'split': GRADCAM_SPLIT,
                'sample_index': index,
                'filename': filename,
                'label': int(y),
                'group': int(group),
                'attribute': int(attr),
                'predicted_class': pred,
                'predicted_probability': float(probs[0, pred].detach().item()),
                'target': 'predicted_class_logit',
                'artifact': stem,
            })
    finally:
        hook.remove()
        activations.clear()

    write_json(GRADCAM_ROOT / 'samples.json', manifest)
    log(f'[Grad-CAM] saved {len(manifest)} sample(s) to {GRADCAM_ROOT}')
    return manifest


gradcam_manifest = save_selected_gradcam()

# Mark the run fully complete after optional Grad-CAM.
summary = json.loads((RUN_ROOT / 'summary.json').read_text(encoding='utf-8'))
summary['status'] = 'complete'
summary['gradcam_samples_saved'] = len(gradcam_manifest)
summary['elapsed_seconds'] = time.monotonic() - RUN_START_TIME
write_json(RUN_ROOT / 'summary.json', summary)

## Experiment switches and multi-seed aggregation

For **all-sample weighting**:

```python
RATING_MODE = "all"
```

For the older **misclassified-only gating**:

```python
RATING_MODE = "misclassified"
CORRECT_WEIGHT = 1.0
```

The trajectory metrics saved in `plots/data/metrics.csv` are clean **unweighted evaluation metrics** after each epoch. The weighted training objective is kept separately in `plots/data/history.csv`.

Use the companion `plot_final_classifier_multiseed.py` script to aggregate completed seeds:

```bash
python plot_final_classifier_multiseed.py \
  --runs /path/to/seed0_run /path/to/seed1_run /path/to/seed2_run \
  --output /path/to/multiseed_summary
```

It aggregates exact recorded epochs only, reports mean ± seed SD, and never performs training or test-based model selection.